In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/ristek-datathon-2022/unlabeled.csv
/kaggle/input/ristek-datathon-2022/sample_submission.csv
/kaggle/input/ristek-datathon-2022/train.csv
/kaggle/input/ristek-datathon-2022/test.csv
/kaggle/input/ristek-datathon-2022/media/Image/ENxfP4KVUAAcBgY.jpg
/kaggle/input/ristek-datathon-2022/media/Image/ERmtSP3XYAI1F1S.jpg
/kaggle/input/ristek-datathon-2022/media/Image/ENL8ySWU4AAYyPD.jpg
/kaggle/input/ristek-datathon-2022/media/Image/ENLO-FfVAAE_QbS.jpg
/kaggle/input/ristek-datathon-2022/media/Image/EOEUS2HVAAAGJ6m.jpg
/kaggle/input/ristek-datathon-2022/media/Image/ERlrAJYUwAAy7r_.jpg
/kaggle/input/ristek-datathon-2022/media/Image/ENc3wKTU0AA1KUQ.jpg
/kaggle/input/ristek-datathon-2022/media/Image/EQzXprHVUAA2F7V.jpg
/kaggle/input/ristek-datathon-2022/media/Image/ENK2epKU4AALRt5.jpg
/kaggle/input/ristek-datathon-2022/media/Image/ENW2TOLU8AEwPJw.jpg
/kaggle/input/ristek-datathon-2022/media/Image/EN1rpdTVAAASEGr.jpg
/kaggle/input/ristek-datathon-2022/media/Image/ENf_a1oU8AAKEF_.jpg
/

In [2]:
import torch, torchvision
import matplotlib.pyplot as plt
import json
import cv2
import numpy as np
from copy import deepcopy
from glob import glob
import pandas as pd
import pickle
from sklearn.model_selection import train_test_split
import gc

/opt/conda/lib/python3.10/site-packages/requests/__init__.py:109: RequestsDependencyWarning: urllib3 (2.1.0) or chardet (None)/charset_normalizer (3.3.2) doesn't match a supported version!
  warnings.warn(


In [3]:
def process_image(df):
    images = []
    texts = []
    for index, row in df.iterrows():
        img = plt.imread(f'/kaggle/input/ristek-datathon-2022/media/{row["media"]}')

        #       # Detectron expects BGR images
        #       img_bgr = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
        images.append(img)

        text = row["text"]
        texts.append(text)
    return images, texts

In [4]:
def path_expander(row):
    img_dir = '/kaggle/input/ristek-datathon-2022/media/Image/' + row
    return img_dir

In [5]:
def stratified_sample_df(df, col, n_samples):
    sample_train, drop_train = train_test_split(df, train_size=n_samples, stratify=df["label"], shuffle=True, random_state=42)
    return sample_train, drop_train

In [6]:
train_df = pd.read_csv(f"/kaggle/input/ristek-datathon-2022/train.csv")
test_df = pd.read_csv(f"/kaggle/input/ristek-datathon-2022/test.csv")

train_df.head()

,created_at,id,user_id,user_name,url,text,media,label
0,2020-02-23 04:06:39+00:00,1231430140824973313,133931409,r0b1 sur1a (黄玉春),https://twitter.com/R0b1Sur1a/status/123143014...,Akhirnya sampai juga setelah menerobos banjir....,ERbrKgFU4AAnADb.jpg,0
1,2020-01-05 01:46:46+00:00,1213637932411580417,253063316,Beradaptasi di Era Pandemi ☀️,https://twitter.com/MarikaRahman_/status/12136...,"Kemekes RI, IDI Banten, IBI Banten dan PPNI Ba...",ENe1HUVUEAANLFT.jpg,0
2,2020-01-18 06:22:09+00:00,1218418277946396673,64318803,rywyu,https://twitter.com/rywyu/status/1218418277946...,Dikarenakan cikini rada2 banjir tdi pagi hingg...,EOiw80RU0AcJ4CI.jpg,0
3,2020-02-22 23:38:00+00:00,1231362534717837313,17383917,ICALIZERS,https://twitter.com/icalizers/status/123136253...,#sperma TT dikala warga jakarta sedang prihati...,ERatrt9UYAAtLMI.jpg,0
4,2019-12-17 10:54:31+00:00,1206890412574568449,3102973556,AN,https://twitter.com/lokbin103/status/120689041...,"KUIS!\n\nJakarta banjir parah hari ini, pertan...",EL-8Z-vUcAIiIlV.jpg,0


In [7]:
train_df['img_dir'] = train_df['media'].apply(path_expander)
test_df['img_dir'] = test_df['media'].apply(path_expander)

In [8]:
train_df = train_df[['img_dir','label','text']]
test_df = test_df[['img_dir','text']]

In [9]:
print(f"Before sample shape: {train_df.shape}")
train_df, val_df = stratified_sample_df(train_df, "label", 0.8)
print(f"After sample shape: {train_df.shape}")

Before sample shape: (1518, 3)
After sample shape: (1214, 3)


In [10]:
# from IPython.display import Image, display
# pil_img = Image(filename='/kaggle/input/ristek-datathon-2022/media/Image/'+train_df.iloc[0]['media'])
# display(pil_img)

In [11]:
pip install --upgrade pip


In [12]:
pip install autogluon

In [13]:
pip install boto3 --upgrade

In [14]:
from autogluon.multimodal import MultiModalPredictor

predictor = MultiModalPredictor(label='label', problem_type='binary',eval_metric='f1',validation_metric='f1')
predictor.fit(
    train_data=train_df,
    tuning_data=val_df,
    hyperparameters={"optimization.learning_rate": 1e-5,'optimization.patience':3, "model.hf_text.checkpoint_name": "indolem/indobertweet-base-uncased"}
)

GPU Count: 1
GPU Count to be Used: 1
GPU 0 Name: Tesla P100-PCIE-16GB
GPU 0 Memory: 0.11GB/16.0GB (Used/Total)

INFO: Using 16bit Automatic Mixed Precision (AMP)
INFO: GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO: IPU available: False, using: 0 IPUs
INFO: HPU available: False, using: 0 HPUs
INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO: 
  | Name              | Type                | Params
----------------------------------------------------------
0 | model             | MultimodalFusionMLP | 207 M 
1 | validation_metric | BinaryF1Score       | 0     
2 | loss_func         | CrossEntropyLoss    | 0     
----------------------------------------------------------
207 M     Trainable params
0         Non-trainable params
207 M     Total params
830.511   Total estimated model params size (MB)


INFO: Epoch 0, global step 4: 'val_f1' reached 0.15556 (best 0.15556), saving model to '/kaggle/working/AutogluonModels/ag-20240419_162801/epoch=0-step=4.ckpt' as top 3


INFO: Epoch 0, global step 10: 'val_f1' reached 0.20690 (best 0.20690), saving model to '/kaggle/working/AutogluonModels/ag-20240419_162801/epoch=0-step=10.ckpt' as top 3


INFO: Epoch 1, global step 14: 'val_f1' reached 0.50000 (best 0.50000), saving model to '/kaggle/working/AutogluonModels/ag-20240419_162801/epoch=1-step=14.ckpt' as top 3


INFO: Epoch 1, global step 20: 'val_f1' reached 0.59130 (best 0.59130), saving model to '/kaggle/working/AutogluonModels/ag-20240419_162801/epoch=1-step=20.ckpt' as top 3


...

INFO: Epoch 4, global step 44: 'val_f1' reached 0.66667 (best 0.66667), saving model to '/kaggle/working/AutogluonModels/ag-20240419_162801/epoch=4-step=44.ckpt' as top 3


INFO: Epoch 4, global step 50: 'val_f1' reached 0.69118 (best 0.69118), saving model to '/kaggle/working/AutogluonModels/ag-20240419_162801/epoch=4-step=50.ckpt' as top 3


INFO: Epoch 5, global step 54: 'val_f1' reached 0.69118 (best 0.69118), saving model to '/kaggle/working/AutogluonModels/ag-20240419_162801/epoch=5-step=54.ckpt' as top 3


INFO: Epoch 5, global step 60: 'val_f1' reached 0.68148 (best 0.69118), saving model to '/kaggle/working/AutogluonModels/ag-20240419_162801/epoch=5-step=60.ckpt' as top 3
Start to fuse 3 checkpoints via the greedy soup algorithm.


AutoMM has created your model. 🎉🎉🎉

To load the model, use the code below:
    ```python
    from autogluon.multimodal import MultiModalPredictor
    predictor = MultiModalPredictor.load("/kaggle/working/AutogluonModels/ag-20240419_162801")
    ```

If you are not satisfied with the model, try to increase the training time, 
adjust the hyperparameters (https://auto.gluon.ai/stable/tutorials/multimodal/advanced_topics/customization.html),
or post issues on GitHub (https://github.com/autogluon/autogluon/issues).




In [15]:
predictor = MultiModalPredictor.load('/kaggle/working/AutogluonModels/ag-20240419_152051/epoch=3-step=34.ckpt')

AssertionError: '/kaggle/working/AutogluonModels/ag-20240419_152051/epoch=3-step=34.ckpt' must be an existing directory.

In [16]:
result = predictor.predict(test_df)

In [17]:
result.value_counts()

label
0    785
1    226
Name: count, dtype: int64

In [18]:
submission = pd.read_csv('/kaggle/input/ristek-datathon-2022/sample_submission.csv')

In [19]:
submission.head()

,id,label
0,1213309429254250496,0
1,1216196702647939072,0
2,1216296533735309312,0
3,1234385081482956800,0
4,1214853229567041537,0


In [20]:
submission['label'] = result

In [21]:
submission.to_csv('autogluon3.csv', index=False)